# Tutorial 2: Training

This tutorial covers training the deepSCENIC model to learn gene regulatory networks from multiome data.

## Learning Objectives

After completing this tutorial, you will be able to:

- Register a genome for sequence extraction
- Understand training configuration options and when to use them
- Configure training for your specific dataset and hardware
- Train a deepSCENIC model with VAE + sequence components
- Save and load trained models for later use
- (Optional) Use a custom sequence model instead of Enformer
- (Optional) Pretrain sequence models separately
- (Optional) Fine-tune E2 matrix for transfer learning

## Setup

First, let's import the required packages and check GPU availability:

In [2]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

deepSCENIC version: 0.1.0
Using device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


```{note}
Training deepSCENIC requires a GPU with at least 16-32GB VRAM due to the Enformer model. Training on CPU is technically possible but extremely slow (hours vs minutes per epoch). See the "Custom Sequence Models" section below if you need a lighter alternative.
```

## Load Preprocessed Data

We'll use the preprocessed MuData from Tutorial 1. The data must contain:
- RNA and ATAC modalities with matching cells
- TF annotations (`is_tf` column in `rna.var`)
- Region-to-gene search space (`r2g` in `uns`)
- Train/test split (`split` column in `obs` and `var`)

In [3]:
import os

# Set your data directory
data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "Data directory not found"

# Load preprocessed MuData
mdata = ds.read(data_dir + "preprocessed_data.h5mu")
print(
    f"Cells: {mdata.n_obs} ({(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test)"
)
print(f"Genes: {mdata['rna'].n_vars} ({mdata['rna'].var['is_tf'].sum()} TFs)")
print(f"Regions: {mdata['atac'].n_vars}")
mdata

deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells, 10421 genes, 281820 regions


Cells: 936 (748 train, 188 test)
Genes: 10421 (1288 TFs)
Regions: 281820


MuData object with n_obs × n_vars = 936 × 292241
  obs:	'split'
  uns:	'r2g'
  2 modalities
    rna:	936 x 10421
      var:	'n_cells', 'is_tf', 'chromosome', 'tss', 'split'
      uns:	'log1p'
      layers:	'log_norm'
    atac:	936 x 281820
      var:	'chromosome', 'start', 'end', 'split'

## Register Genome

deepSCENIC extracts DNA sequences from genomic regions to learn TF binding patterns. Before training, you must register a genome FASTA file that matches your data's reference assembly.

In [10]:
try:
    genome = ds.get_genome()
    print(f"Genome already registered: {genome.name}")
except RuntimeError:
    ds.register_genome("../../../../data/genomes/hg38.fa")

Genome already registered: hg38


## Understanding the Architecture

deepSCENIC learns gene regulatory networks through interconnected components:

```
                    DNA Sequence
                         │
                    ┌────▼────┐
                    │ Enformer │  (Pretrained DNA language model)
                    └────┬────┘
                         │
                    ┌────▼────┐
                    │ TF2rNet  │  (Maps embeddings → TF binding)
                    └────┬────┘
                         │
                    ┌────▼────┐
                    │   E1    │  TF → Region weights (learned)
                    └────┬────┘
                         │
    TF Expression ──┬────▼────┐
         │          │   VAE   │  (Encodes TF activity)
         │          └────┬────┘
         │               │
         │          ┌────▼────┐
         │          │   E2    │  Region → Gene weights (learned)
         │          └────┬────┘
         │               │
         └───────────────▼
              Gene Expression (reconstructed)
```

The model learns two key regulatory matrices:
- **E1 (TF→Region)**: Which TFs bind to which regulatory regions
- **E2 (Region→Gene)**: Which regions regulate which genes

Together, these matrices form the Gene Regulatory Network (GRN).

## Training Stages

deepSCENIC training involves multiple phases with warmup periods:

```
Epoch:    0        10        50       100
          │         │         │         │
Phase 1:  ├─────────┤         │         │
          │  VAE    │         │         │
          │  only   │         │         │
          │         │         │         │
Phase 2:  │         ├─────────┤         │
          │         │ VAE +   │         │
          │         │ TF2rNet │         │
          │         │         │         │
Phase 3:  │         │         ├─────────┤
          │         │         │  Full   │
          │         │         │ (frozen │
          │         │         │ seq.)   │
          └─────────┴─────────┴─────────┘
```

- **Phase 1 (warmup_vae)**: Train only the VAE to learn cell representations
- **Phase 2 (warmup_vae → warmup_grn)**: Joint VAE + sequence model training
- **Phase 3 (after warmup_grn)**: Freeze sequence models, refine E1/E2 matrices

This phased approach stabilizes training by letting each component learn sequentially.

## Understanding Training Configuration

Training is configured through the `ds.tl.train()` function, which accepts both direct parameters and a `TrainingConfig` object. Understanding the key parameters helps you adapt training to your specific dataset and hardware.

### Two Ways to Configure Training

1. **Direct parameters** (simple): Pass common settings directly to `ds.tl.train()`
2. **TrainingConfig** (advanced): Use a config object for full control over all settings

When you provide both, **direct parameters override config values**. This lets you use a config as a baseline and tweak specific settings.

### Understanding the Two Batch Sizes

deepSCENIC uses two separate data streams during training, each with its own batch size:

| Parameter | Controls | Default | What it loads |
|-----------|----------|---------|---------------|
| `batch_size` | Cell dataloader | 64 | RNA + ATAC expression per cell |
| `seq_batch_size` | Sequence dataloader | 1000 | DNA sequences for TF binding |

**Why two dataloaders?**

The **cell dataloader** samples cells for the VAE. Each batch contains `batch_size` cells with their RNA and ATAC profiles. This trains the VAE to reconstruct gene expression from TF activity.

The **sequence dataloader** samples genomic regions for the TF2rNet. Each batch contains `seq_batch_size` DNA sequences that are processed through Enformer to update the E1 matrix (TF-to-region binding predictions).

```{tip}
The sequence batch size can be much larger (1000 vs 64) because DNA sequences are processed independently, while cell batches need to fit the full VAE computation in GPU memory.
```

### Key Training Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| **Computational** ||||
| `epochs` | int | 100 | Total training epochs |
| `batch_size` | int | 64 | Cells per batch |
| `seq_batch_size` | int | 1000 | Sequences per batch |
| `device` | str | "cuda" | Training device |
| **Learning rates** ||||
| `lr` | float | 1e-3 | Base learning rate (maps to `lr_vae`) |
| `lr_vae` | float | 1e-3 | VAE learning rate (via config) |
| `lr_tf2rnet` | float | 1e-3 | Sequence model learning rate (via config) |
| **Warmup phases** ||||
| `warmup_vae` | int | 10 | Epochs before enabling TF2rNet |
| `warmup_grn` | int | 50 | Epochs before freezing sequence models |
| **Regularization** ||||
| `beta` | float | 1e-2 | KL divergence weight |
| `alpha` | float | 1e-2 | E1 sparsity weight |
| `gamma` | float | 1.0 | E2 sparsity weight |

```{note}
Parameters available as direct arguments to `ds.tl.train()`: `epochs`, `batch_size`, `seq_batch_size`, `lr`, `device`, `warmup_vae`, `warmup_grn`. All others require `TrainingConfig`.
```

### ds.tl.train() vs TrainingConfig

The `ds.tl.train()` function exposes the most commonly adjusted parameters directly:

```python
# Direct parameters (simple approach)
model = ds.tl.train(
    mdata,
    epochs=100,
    batch_size=64,
    lr=1e-3,           # Sets lr_vae in the config
    warmup_vae=10,
    warmup_grn=50,
    device="cuda",
)
```

For advanced settings (loss functions, separate learning rates, custom models), use `TrainingConfig`:

```python
from deepscenic.tl import TrainingConfig

# Full configuration (advanced approach)
config = TrainingConfig(
    epochs=1000,
    batch_size=64,
    seq_batch_size=1000,
    lr_vae=1e-3,
    lr_tf2rnet=1e-4,    # Separate learning rate for sequence model
    beta=1e-2,
    alpha=1e-2,
    gamma=1.0,
    loss_rna="mse",     # Options: 'mse', 'mae', 'cosine'
    loss_atac="mse",    # Options: 'mse', 'mae', 'bce', 'cosine'
)

model = ds.tl.train(mdata, config=config, device="cuda")
```

You can also combine both approaches - direct parameters override config values:

```python
# Use config as baseline, override specific settings
config = TrainingConfig(epochs=500, lr_vae=1e-4)
model = ds.tl.train(mdata, config=config, lr=1e-3)  # lr=1e-3 overrides lr_vae
```

## Configure and Run Training

For most datasets, the default parameters work well. Here's a simple configuration:

In [ ]:
# Simple training configuration
# For demonstration, we use 100 epochs. For real training, use 500-1000 epochs.

training_params = {
    "epochs": 100,
    "batch_size": 64,
    "lr": 1e-4,
    "warmup_vae": 10,
    "warmup_grn": 50,
    "device": device,
}

print("Training configuration:")
for key, value in training_params.items():
    print(f"  {key}: {value}")

## Train the Model

Now we're ready to train. The `ds.tl.train()` function:
1. Initializes the VAE, TF2rNet, and Enformer models
2. Computes initial E1 matrix from sequences (may take a few minutes)
3. Runs the training loop with the warmup schedule
4. Returns a `DeepSCENICModel` containing all trained components

```{note}
The first training run downloads the pretrained Enformer model (~1GB). Subsequent runs use the cached version. Initial E1 computation also takes several minutes as it processes all genomic sequences.
```

In [ ]:
# Train the deepSCENIC model
model = ds.tl.train(
    mdata,
    epochs=training_params["epochs"],
    batch_size=training_params["batch_size"],
    lr=training_params["lr"],
    warmup_vae=training_params["warmup_vae"],
    warmup_grn=training_params["warmup_grn"],
    device=training_params["device"],
)

print("\nTraining complete!")
print(f"Model contains {len(model.tf_names)} TFs, {len(model.gene_names)} genes, {len(model.region_names)} regions")

## Monitor Training Progress

deepSCENIC supports logging backends for tracking training metrics:

```python
# TensorBoard logging
model = ds.tl.train(
    mdata,
    epochs=100,
    logger="tensorboard",
    log_dir="./runs/deepscenic_experiment",
    device=device,
)
```

Then launch TensorBoard in a separate terminal:
```bash
tensorboard --logdir ./runs
```

Alternatively, use Weights & Biases with `logger="wandb"`.

## Save and Load Models

Trained models can be saved for later use. The saved file includes:
- All model weights (VAE, TF2rNet, sequence model)
- The E1 matrix cache
- Training configuration
- Feature names (TFs, genes, regions)

```{note}
Model files are large (~1GB+) when using Enformer. Models trained with custom sequence models will be smaller.
```

In [ ]:
# Save the trained model
model.save(data_dir + "deepscenic_model.pt")
print(f"Model saved to {data_dir}deepscenic_model.pt")

In [ ]:
# Load a saved model in future sessions
loaded_model = ds.tl.load_model(data_dir + "deepscenic_model.pt", device=device)
print(f"Loaded model with {len(loaded_model.tf_names)} TFs")

## Custom Sequence Models

By default, deepSCENIC uses Enformer as its sequence encoder. However, you can provide your own PyTorch module instead. This is useful when:

- You want a lighter model that doesn't require 16GB+ VRAM
- You have a custom architecture tailored to your genomic context
- You want to experiment with different sequence representations

### Model Requirements

Your custom sequence model must satisfy these requirements:

| Requirement | Details |
|-------------|---------|
| **Input shape** | `(batch, seq_len, 4)` - one-hot encoded DNA sequences |
| **Output shape** | `(batch, bottleneck_size * emb_len)` - flattened embeddings |
| **Config sync** | Set matching `seq_len`, `bottleneck_size`, `emb_len` in config |

The three key parameters control the model's interface:

- **`seq_len`**: Input DNA sequence length in base pairs. Default is 640 for Enformer compatibility. Set this to match your model's expected input length.
- **`bottleneck_size`**: Embedding dimension per position. Default is 3072 (Enformer's embedding size). This is the number of features your model outputs per position.
- **`emb_len`**: Number of output positions. Default is 5 (Enformer's target_length). Combined with bottleneck_size, this determines your model's total output: `bottleneck_size * emb_len`.

In [ ]:
import torch.nn as nn

from deepscenic.tl import TrainingConfig


# Example: A lightweight convolutional sequence model
class LightweightSequenceModel(nn.Module):
    """Lightweight sequence encoder for reduced GPU memory."""

    def __init__(self, seq_len=640, bottleneck_size=256, emb_len=60):
        super().__init__()
        # Store dimensions for verification
        self.seq_len = seq_len
        self.bottleneck_size = bottleneck_size
        self.emb_len = emb_len
        self.output_dim = bottleneck_size * emb_len

        # Simple conv architecture
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(64, bottleneck_size, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(emb_len)

    def forward(self, x):
        """Transform one-hot DNA to flattened embeddings."""
        # x: (batch, seq_len, 4)
        x = x.transpose(1, 2)  # (batch, 4, seq_len)
        x = self.conv(x)  # (batch, bottleneck_size, seq_len)
        x = self.pool(x)  # (batch, bottleneck_size, emb_len)
        return x.flatten(1)  # (batch, bottleneck_size * emb_len)


# Create model instance
custom_model = LightweightSequenceModel(
    seq_len=640,
    bottleneck_size=256,
    emb_len=60,
)

# Configure training with matching dimensions
config = TrainingConfig(
    sequence_model=custom_model,
    seq_len=640,  # Must match model's expected input
    bottleneck_size=256,  # Must match model's output channels
    emb_len=60,  # Must match model's output length
    epochs=100,
)

print(f"Custom model output: {custom_model.output_dim} = {config.bottleneck_size} x {config.emb_len}")

### Training with Custom Models

```python
# Train with custom sequence model
model = ds.tl.train(mdata, config=config, device=device)

# Save the model
model.save("custom_model.pt")
```

### Loading Custom Models

When loading a model trained with a custom sequence model, you must provide an instance of the same architecture:

```python
# Recreate the model architecture (same as during training)
custom_model = LightweightSequenceModel(seq_len=640, bottleneck_size=256, emb_len=60)

# Load with the custom model
loaded_model = ds.tl.load_model(
    "custom_model.pt",
    device=device,
    sequence_model=custom_model,  # Required for custom models
)
```

```{warning}
The model architecture must exactly match what was used during training. Only the weights are saved in the checkpoint file, not the model class definition.
```

## Optional: Pretraining Sequence Models

For large datasets, pretraining the sequence models (Enformer + TF2rNet) can improve convergence. Pretraining learns TF binding patterns from DNA sequences before joint training with the VAE.

```{note}
Pretraining is **optional**. It's most beneficial for:
- Large datasets (>10,000 cells)
- Datasets with many accessible regions (>100,000)
- When you want to reuse sequence models across multiple experiments
```

In [ ]:
# Optional: Pretrain sequence models
# pretrained = ds.tl.pretrain(
#     mdata,
#     epochs=100,
#     batch_size=256,
#     lr=1e-4,
#     device=device,
# )
# pretrained.save(data_dir + "pretrained.pt")

# Then use in training:
# model = ds.tl.train(
#     mdata,
#     pretrained_model=pretrained,
#     epochs=100,
#     device=device,
# )

## Optional: Fine-tuning E2 Matrix

Fine-tuning trains only the E2 (region→gene) matrix while keeping all other components frozen. This is useful for:
- **Transfer learning**: Adapting a trained model to new data
- **Domain adaptation**: When cell types differ between training and target data

```{note}
Fine-tuning uses a very small learning rate (1e-6) and many epochs (10,000) to gradually adjust E2 weights without disrupting learned representations.
```

In [ ]:
# Optional: Fine-tune E2 matrix
# model = ds.tl.finetune(
#     model,
#     mdata,
#     epochs=10000,
#     lr=1e-6,
#     reinit_e2=True,  # Reset E2 to near-zero before finetuning
#     device=device,
# )

## Complete Workflow Example

Here's a complete training workflow with all optional stages:

In [ ]:
# Complete deepSCENIC training workflow
#
# # 1. Register genome
# ds.register_genome("/path/to/hg38.fa")
#
# # 2. Load preprocessed data
# mdata = ds.read("preprocessed_data.h5mu")
#
# # 3. (Optional) Pretrain sequence models
# pretrained = ds.tl.pretrain(mdata, epochs=100, device="cuda")
# pretrained.save("pretrained.pt")
#
# # 4. Main training
# model = ds.tl.train(
#     mdata,
#     epochs=1000,
#     batch_size=64,
#     lr=1e-4,
#     warmup_vae=10,
#     warmup_grn=50,
#     pretrained_model=pretrained,
#     logger="tensorboard",
#     log_dir="./runs",
#     device="cuda",
# )
#
# # 5. (Optional) Fine-tune for transfer learning
# model = ds.tl.finetune(model, mdata, epochs=10000, lr=1e-6, device="cuda")
#
# # 6. Save final model
# model.save("final_model.pt")

## Next Steps

Now that you have a trained model, you can:

- Diagnose model training and inspect results: {doc}`03_model_diagnosis`
- Extract and analyze the learned GRN: {doc}`04_grn_analysis`
- Create publication-ready visualizations: {doc}`05_visualization`